# Churn Model EDA + Training

This notebook combines the EDA scatter and model training workflow.

In [1]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


## 1) Data Load + Correlation Scatter (Top 2 Features)

In [ ]:
data_path = Path('data') / 'insurance_policyholder_churn_synthetic.csv'
df = pd.read_csv(data_path)

target_col = 'churn_flag'
drop_cols = ['customer_id', 'as_of_date', 'churn_type', 'churn_probability_true']
drop_cols = [c for c in drop_cols if c in df.columns]

numeric_df = df.drop(columns=drop_cols).select_dtypes(include=['number'])
corr = numeric_df.corr(numeric_only=True)[target_col].drop(labels=[target_col])
top2 = corr.abs().sort_values(ascending=False).head(2).index.tolist()
x_col, y_col = top2

fig, ax = plt.subplots(figsize=(6.5, 5), dpi=140)
for label, color in [(0, '#4C78A8'), (1, '#F58518')]:
    subset = df[df[target_col] == label]
    ax.scatter(subset[x_col], subset[y_col], s=12, alpha=0.45, label=f'{target_col}={label}', color=color)

ax.set_title('Top Correlated Features vs Target')
ax.set_xlabel(x_col)
ax.set_ylabel(y_col)
ax.legend()
fig.tight_layout()

out_path = Path('model') / 'target_corr_scatter.png'
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, bbox_inches='tight')
plt.close(fig)

print(f'Saved plot to: {out_path}')
print(f'Top correlated features: {x_col}, {y_col}')


## 2) Training Functions

In [ ]:
def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]

    cat_pipe = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]
    )
    num_pipe = Pipeline(
        steps=[('imputer', SimpleImputer(strategy='median'))]
    )

    return ColumnTransformer(
        transformers=[
            ('cat', cat_pipe, cat_cols),
            ('num', num_pipe, num_cols),
        ],
        remainder='drop',
    )


def add_engineered_features(X: pd.DataFrame) -> pd.DataFrame:
    result = X.copy()
    result['premium_increase_shock'] = np.clip(result['premium_change_pct'], 0, None) * (1 + result['num_price_increases_last_3y'])
    result['premium_jump_flag'] = (result['premium_change_pct'] >= 0.12).astype(int)
    result['payment_risk_score'] = result['late_payment_count_12m'] + result['missed_payment_flag'] * 2
    result['service_risk_score'] = result['complaint_flag'] + result['quote_requested_flag'] + result['coverage_downgrade_flag']
    result['engagement_risk_score'] = result['payment_risk_score'] + result['service_risk_score']
    result['tenure_inverse'] = 1 / (result['customer_tenure_months'] + 1)
    result['single_policy_short_tenure'] = ((result['multi_policy_flag'] == 0) & (result['customer_tenure_months'] <= 24)).astype(int)
    result['premium_x_complaint'] = result['premium_jump_flag'] * result['complaint_flag']
    result['premium_x_quote'] = result['premium_jump_flag'] * result['quote_requested_flag']
    result['late_x_quote'] = (result['late_payment_count_12m'] >= 2).astype(int) * result['quote_requested_flag']
    result['downgrade_x_quote'] = result['coverage_downgrade_flag'] * result['quote_requested_flag']
    result['monthly_payment_flag'] = (result['payment_frequency'] == 'Monthly').astype(int)
    result['monthly_x_premium_jump'] = result['monthly_payment_flag'] * result['premium_jump_flag']
    result['auto_or_health'] = result['policy_type'].isin(['Auto', 'Health']).astype(int)
    return result


def fbeta_score_from_pr(precision: np.ndarray, recall: np.ndarray, beta: float) -> np.ndarray:
    beta2 = beta * beta
    return (1 + beta2) * (precision * recall) / (beta2 * precision + recall + 1e-12)


def select_threshold(thresholds, precision, recall, opt_metric='f1', min_precision=0.0, min_recall=0.0):
    if opt_metric == 'f2':
        scores = fbeta_score_from_pr(precision, recall, beta=2.0)
    else:
        scores = fbeta_score_from_pr(precision, recall, beta=1.0)

    mask = (precision >= min_precision) & (recall >= min_recall)
    if mask.any():
        idx = int(np.nanargmax(scores * mask))
        return float(thresholds[idx])

    idx = int(np.nanargmax(scores))
    return float(thresholds[idx])


def select_threshold_by_accuracy(y_true, y_prob, thresholds):
    candidate_thresholds = np.unique(np.clip(np.concatenate(([0.0], thresholds, [1.0])), 0.0, 1.0))
    best_threshold = 0.5
    best_accuracy = -1.0

    for threshold in candidate_thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        accuracy = accuracy_score(y_true, y_pred)
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_threshold = float(threshold)

    return best_threshold, best_accuracy


def save_threshold_plot(y_true, y_prob, selected_threshold, out_path):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    thresholds = thresholds.astype(float)
    precision_t = precision[1:]
    recall_t = recall[1:]
    f1 = fbeta_score_from_pr(precision_t, recall_t, beta=1.0)
    f2 = fbeta_score_from_pr(precision_t, recall_t, beta=2.0)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4), dpi=140)
    axes[0].plot(recall, precision, color='#4C78A8')
    axes[0].set_title('PR Curve')
    axes[0].set_xlabel('Recall')
    axes[0].set_ylabel('Precision')

    axes[1].plot(thresholds, recall_t, label='Recall', color='#1F77B4')
    axes[1].plot(thresholds, precision_t, label='Precision', color='#FF7F0E')
    axes[1].plot(thresholds, f1, label='F1', color='#2CA02C', linestyle='--')
    axes[1].plot(thresholds, f2, label='F2', color='#9467BD', linestyle='--')
    axes[1].axvline(selected_threshold, color='red', linestyle=':', label='Selected')
    axes[1].set_title('Threshold vs Metrics')
    axes[1].set_xlabel('Threshold')
    axes[1].set_ylabel('Score')
    axes[1].legend(fontsize=8)

    axes[2].scatter(thresholds, f2, s=12, color='#7F7F7F')
    axes[2].axvline(selected_threshold, color='red', linestyle=':')
    axes[2].set_title('F2 by Threshold')
    axes[2].set_xlabel('Threshold')
    axes[2].set_ylabel('F2')

    fig.tight_layout()
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)


## 3) Train Model

In [ ]:
target_col = 'churn_flag'
drop_cols = ['customer_id', 'as_of_date', 'churn_type', 'churn_probability_true']
drop_cols = [c for c in drop_cols if c in df.columns]

y = df[target_col].astype(int).to_numpy()
X = df.drop(columns=drop_cols + [target_col])
X = add_engineered_features(X)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)

preprocessor = build_preprocessor(X_train)
model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=None,
    max_iter=400,
    l2_regularization=0.0,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    random_state=42,
)

pipeline = Pipeline(
    steps=[('preprocess', preprocessor), ('model', model)]
)

pipeline.fit(X_train, y_train)

y_val_prob = pipeline.predict_proba(X_val)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, y_val_prob)
thresholds = thresholds.astype(float)
precision_t = precision[1:]
recall_t = recall[1:]

threshold = select_threshold(
    thresholds=thresholds,
    precision=precision_t,
    recall=recall_t,
    opt_metric='f2',
    min_precision=0.3,
    min_recall=0.85,
)

final_preprocessor = build_preprocessor(X_train_full)
final_pipeline = Pipeline(
    steps=[
        ('preprocess', final_preprocessor),
        ('model', HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_depth=None,
            max_iter=400,
            l2_regularization=0.0,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            random_state=42,
        )),
    ]
)
final_pipeline.fit(X_train_full, y_train_full)

y_prob = final_pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= threshold).astype(int)

auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
val_acc = accuracy_score(y_val, (y_val_prob >= threshold).astype(int))
val_recall = ((y_val_prob >= threshold).astype(int)[y_val == 1]).mean()
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f'ROC AUC: {auc:.4f}')
print(f'PR AUC: {pr_auc:.4f}')
print(f'Accuracy: {acc:.4f}')
print(f'F1: {f1:.4f}')
print(f'Validation Accuracy: {val_acc:.4f}')
print(f'Validation Recall: {val_recall:.4f}')
print(f'Selected threshold (F2 / Recall-focused): {threshold:.4f}')
print('Confusion Matrix:')
print(cm)
print('Classification Report:')
print(classification_report(y_test, y_pred, digits=4))

save_threshold_plot(
    y_true=y_test,
    y_prob=y_prob,
    selected_threshold=threshold,
    out_path=Path('model') / 'threshold_analysis_new.png',
)
print('Saved threshold plot to: model/threshold_analysis_new.png')

joblib.dump(final_pipeline, Path('model') / 'churn_model_new.pkl')
joblib.dump(threshold, Path('model') / 'threshold_new.pkl')
print('Saved model to: model/churn_model_new.pkl')
print('Saved threshold to: model/threshold_new.pkl')
